# Zaimi et al. (2018), AxonDeepSeg, tested on dandiset 001436

**Paper.** Zaimi A, Wabartha M, Herman V, Antonsanti P-L, Perone CS, Cohen-Adad J (2018).
AxonDeepSeg: automatic axon and myelin segmentation from microscopy data using convolutional
neural networks. *Scientific Reports* 8, 3816. doi:10.1038/s41598-018-22181-4.

**Dataset.** DANDI:001436, "TEM Images of Corpus Callosum in Control and Cuprizone-Intoxicated
Mice with Axon and Myelin Segmentations", version **0.250512.1625**
(doi:10.48324/dandi.001436/0.250512.1625). The paper cites no version of the dandiset (the dandiset
was created in 2025, after the paper). 0.250512.1625 is the only published version, and the
draft (modified 2026-09-23) has the same 660 assets and size, so the most recent published
version is pinned. The dataset is a BIDS microscopy dataset of png images, not NWB; the files
are read from the archive into memory.

**Text the claims were written from.** The version of record (PMC5830647), with Methods,
figure legends and Tables 1 to 3; the paper has no supplement. Claims file:
`claims/001436/10.1038-s41598-018-22181-4.yaml`.

**Outside inputs.** (1) The paper's released TEM model, `default_TEM_model_v1`, at commit
7f6013221b2a862e107a7e23f3c8662eba3ba64b of https://github.com/neuropoly/axondeepseg, with the
inference code at the same commit (claims `data: with_inputs`). (2) The folder names of the NYU
mouse component (OSF node `adrcs`) of the White Matter Microscopy Database
(https://osf.io/yp4qg/, doi:10.17605/OSF.IO/YP4QG), which the paper's Data availability
statement names, to label each mouse as control or cuprizone (`data: with_labels`, tested by
the link claim `osf-group-labels-link`). (3) The macaque TEM images and masks of ref. 7 in the
same database (OSF node `yp4qg`, folder MontrealNeurologicalInstitute/histology/Macaque), for
the two macaque claims (`data: elsewhere`).

| Claim | Tested here |
|---|---|
| osf-group-labels-link (added by this attempt) | tested |
| table3-tem-mice-accuracy | tested |
| table3-tem-mice-dice-detection | tested |
| myelin-dice-below-axon-dice | tested (mouse images only) |
| sensitivity-above-93 | tested (mouse images only) |
| table3-sem-rat-accuracy | blocked, no SEM images in the dandiset |
| table3-sem-human-accuracy | blocked, no SEM images in the dandiset |
| table3-tem-macaque-accuracy | tested on the macaque images of the White Matter Microscopy Database (data elsewhere) |
| table3-other-samples-dice-detection | blocked, no SEM images in the dandiset or elsewhere (macaque values shown) |
| macaque-myelin-underestimated | tested on the macaque images of the White Matter Microscopy Database (data elsewhere) |
| full-slice-processing-time | blocked, no SEM slice in the dandiset |
| fig7-corticospinal-tract | blocked, no rat spinal cord in the dandiset |
| fig7-larger-axons-periphery | blocked, as above |
| fig7-tract-diameters | blocked, as above |
| fig7-gratio-range | blocked, as above |

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import itertools
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import axondeepseg_tem as ads

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dandiset", ads.DANDISET_ID, "version", ads.DANDISET_VERSION)
print("AxonDeepSeg commit", ads.ADS_COMMIT)

## Data

The archive's metadata, the file list and the sidecar files. Each mouse has one sidecar
(`sub-<mouse>_TEM.json`) giving the pixel size.

In [ ]:
meta = ads.version_metadata()
print(meta["name"])
print("assets", meta["assetsSummary"]["numberOfFiles"], "bytes", meta["assetsSummary"]["numberOfBytes"],
      "subjects", meta["assetsSummary"]["numberOfSubjects"], "samples (archive summary)",
      meta["assetsSummary"]["numberOfSamples"])
print(meta["description"])

assets = ads.list_assets()
kind = assets.path.str.replace(r"sub-nyuMouse\d+", "sub-*", regex=True).str.replace(r"sample-\d+", "sample-*", regex=True)
print(kind.value_counts().to_string())
for p in ["README", "CHANGES", "participants.tsv", "samples.json", "derivatives/labels/dataset_description.json"]:
    print("=====", p)
    print(ads.read_bytes(assets.set_index("path").asset_id[p]).decode())

In [ ]:
table = ads.sample_table(assets)
sidecars = {s: json.loads(ads.read_bytes(a)) for s, a in table.groupby("subject").sidecar.first().items()}
print(pd.DataFrame(sidecars).T.astype(str).drop_duplicates().to_string())
table["pixel_um"] = table.subject.map({s: d["PixelSize"][0] for s, d in sidecars.items()})
samples_tsv = pd.read_csv(pd.io.common.BytesIO(ads.read_bytes(assets.set_index("path").asset_id["samples.tsv"])), sep="\t")
listed = set(zip(samples_tsv.participant_id, samples_tsv.sample_id))
found = set(zip(table.subject, table["sample"]))
print("images", len(table), "with all three masks", int(table[["axon", "myelin", "axonmyelin"]].notna().all(axis=1).sum()))
print("samples.tsv rows", len(samples_tsv), "| listed but no image", sorted(listed - found), "| image not listed", sorted(found - listed))
print("images per mouse:", table.groupby("subject").size().to_dict())

### Group labels from the White Matter Microscopy Database

The dandiset does not say which mice are controls and which received cuprizone, though its
description says it holds both. The claims name, as one reading of the test set, the control
mice only. The paper's Data availability statement names the White Matter Microscopy Database;
its NYU component (OSF node `adrcs`) holds the raw micrographs of ref. 27 in one folder per diet
group and time point, with one sub-folder per mouse named by the mouse's number. Only the folder
names are used.

In [ ]:
osf = ads.osf_mouse_folders()
print(osf[["group_folder", "mouse_folder", "mouse"]].to_string())

## The model

The paper's TEM model is read from GitHub at the pinned commit. Its network is rebuilt from the
checkpoint (see the helper's docstring), and checked here against the model's own saved
TensorFlow graph (`model.ckpt.meta`), run with `training_phase` False, on the four patches of the
first image.

In [ ]:
weights, config = ads.load_tem_model()
print("variables", len(weights), "parameters", sum(int(np.prod(v.shape)) for v in weights.values()))
print({k: config[k] for k in ["depth", "trainingset_patchsize", "n_classes", "convolution_per_layer", "downsampling", "batch_norm_activate"]})

first = table.iloc[0]
img0, gt0, _ = ads.load_sample(first)
from skimage.transform import rescale
rs0 = rescale((255 - img0.astype(int)).astype(np.uint8), first.pixel_um / ads.WORKING_PIXEL_UM, order=1,
              mode="constant", preserve_range=True, anti_aliasing=False).astype(int)
patches0, pos0 = ads.im2patches_overlap(rs0)
mine = ads.unet_forward(weights, config, patches0)
theirs = ads.original_graph_output(patches0)
print("patches", len(patches0), "positions", pos0)
print("largest absolute difference of the outputs", float(np.abs(mine - theirs).max()),
      "| output range", float(theirs.min()), float(theirs.max()))
print("fraction of pixels with the same class", float((mine.argmax(3) == theirs.argmax(3)).mean()))

## Segmenting every image

Each image is segmented as `segment.py` does for this model (contrast inverted once, 0.01 um
working pixel size, 512 x 512 patches with an overlap of 25 pixels), and, as the variant the
criterion asks for, without the inversion. Scores are computed against the dandiset's masks at
the native pixel size.

In [ ]:
t0 = time.time()
res = ads.run_all(table, weights, config)
res["mouse"] = res.subject.str.extract(r"(\d+)$")[0].astype(int)
print(f"{len(res)} images segmented twice in {(time.time() - t0) / 60:.1f} min")

## Data check

In [ ]:
print("images", len(res), "mice", res.subject.nunique())
print("images per mouse:", res.groupby("subject").size().to_dict())
print("pixel size (um) in the sidecars:", sorted(table.pixel_um.unique()), "| Table 1 of the paper gives 0.002")
print("image shapes:", res.groupby(["height", "width"]).size().to_dict())
print("field of view (um): %.2f x %.2f" % (res.height.iloc[0] * table.pixel_um.iloc[0], res.width.iloc[0] * table.pixel_um.iloc[0]),
      "| Table 1 gives 6 x 9")
print("grey levels: min over images", res.img_min.min(), "max", res.img_max.max(), "| lowest image max", res.img_max.min())
print("axon mask values:", res.axon_values.value_counts().to_dict())
print("myelin mask values:", res.myelin_values.value_counts().to_dict())
print("axonmyelin mask values:", res.axonmyelin_values.value_counts().to_dict())
print("dimensions of the axon mask, myelin mask and image:", res.mask_dims.value_counts().to_dict())
print("axonmyelin pngs with 1 channel (grey) and 2 (grey and alpha):", res.axonmyelin_channels.value_counts().to_dict(),
      "| alpha values:", res.axonmyelin_alpha.dropna().value_counts().to_dict())
print("axonmyelin mask (grey channel) equals axon (255) and myelin (128) masks in", int(res.am_agrees.sum()), "of", len(res), "images")
print("pixels both axon and myelin: total", int(res.axon_myelin_overlap.sum()))
print("ground-truth fractions: axon %.3f to %.3f, myelin %.3f to %.3f" % (res.inv_gt_axon_frac.min(), res.inv_gt_axon_frac.max(),
      res.inv_gt_myelin_frac.min(), res.inv_gt_myelin_frac.max()))
print("ground-truth axon objects per image: %d to %d" % (res.inv_P.min(), res.inv_P.max()))

# Masks in register with their images: myelin is the darkest structure in TEM.
print("grey-level contrast, rest of image minus myelin: masks in place median %.1f (min %.1f); "
      "shifted by 50 px median %.1f (max %.1f); larger in place in %d of %d images" % (
      res.contrast.median(), res.contrast.min(), res.contrast_shifted.median(), res.contrast_shifted.max(),
      int((res.contrast > res.contrast_shifted).sum()), len(res)))

# Images deposited twice.
print("distinct images by content hash:", res.sha1.nunique(), "| distinct ground truths:", res.gt_sha1.nunique())
T = np.asarray(list(res.thumb), float)
T = (T - T.mean(1, keepdims=True)) / T.std(1, keepdims=True)
R = T @ T.T / T.shape[1]
iu = np.triu_indices(len(res), 1)
print("largest thumbnail correlation between two different images: %.3f; pairs above 0.95: %d" % (
      R[iu].max(), int((R[iu] > 0.95).sum())))

# The pixel size of the paper (0.002 um) would give images too small for a 512 x 512 patch.
h002 = np.round(res.height * 0.002 / ads.WORKING_PIXEL_UM).astype(int)
print("height after resampling with 0.002 um: %d to %d px (a patch is 512); with the sidecar's %.5f um: %s" % (
      h002.min(), h002.max(), table.pixel_um.iloc[0], sorted({s[0] for s in res.resampled_shape})))
print("patches per image:", res.n_patches.value_counts().to_dict())

## osf-group-labels-link

> Each mouse of the dandiset is found, by its number, in exactly one diet-group folder of the
> NYU component of the White Matter Microscopy Database, and the deposits agree on what both
> hold (added by this attempt; see the claims file for the criterion).

In [ ]:
grp = osf.groupby("mouse").group_folder.agg(lambda s: sorted(set(s)))
dmice = sorted(res.mouse.unique())
link = pd.DataFrame({"mouse": dmice})
link["osf_groups"] = link.mouse.map(lambda m: grp.get(m, []))
link["n_groups"] = link.osf_groups.map(len)
link["group"] = link.osf_groups.map(lambda g: g[0] if len(g) == 1 else None)
link["control"] = link.group.str.startswith("ctl")
print(link.to_string())
print("dandiset mice matched to exactly one group folder:", int((link.n_groups == 1).sum()), "of", len(link))
print("mice per group folder in the dandiset:", link.group.value_counts().to_dict())
print("mice per group folder on OSF:", osf.groupby("group_folder").mouse.nunique().to_dict())
print("controls:", list(link.mouse[link.control]), "| cuprizone:", list(link.mouse[~link.control]))

In [ ]:
# Content: for each matched mouse whose OSF folder holds full-size micrographs, up to two of them
# (in name order) are compared with every dandiset image by thumbnail correlation.
shape = (int(res.height.iloc[0]), int(res.width.iloc[0]))
rows = []
for _, fr in osf[osf.mouse.isin(dmice)].iterrows():
    files = ads.osf_files(fr.files_url)
    for name, fid, oshape, th in ads.osf_thumbnails(files, shape):
        if th is None:
            rows.append({"osf_folder": fr.mouse_folder, "file": name, "shape": oshape, "note": "smaller than the dandiset images"})
            continue
        b = ads.best_match(th, list(res.thumb))
        i = int(np.argmax(b))
        same = b[(res.mouse == fr.mouse).values]
        other = b[(res.mouse != fr.mouse).values]
        rows.append({"osf_folder": fr.mouse_folder, "file": name, "id": fid, "shape": oshape,
                     "best_image": f"{res.subject[i]} {res['sample'][i]}", "r_best": round(float(b[i]), 3),
                     "best_same_mouse_r": round(float(same.max()), 3), "best_other_mouse_r": round(float(other.max()), 3)})
content = pd.DataFrame(rows)
print(content.to_string())
ok = content.dropna(subset=["r_best"]).copy()
ok["mouse"] = ok.osf_folder.str.extract(r"^(\d+)")[0].astype(int)
ok["best_mouse"] = ok.best_image.str.extract(r"nyuMouse(\d+)")[0].astype(int)
cp = ok[ok.r_best > 0.5]
print("OSF micrographs compared:", len(ok), "from", ok.mouse.nunique(), "mice | with a counterpart (r > 0.5):", len(cp),
      "| counterpart of the same mouse:", int((cp.mouse == cp.best_mouse).sum()),
      "| of another mouse:", int((cp.mouse != cp.best_mouse).sum()))
print("mice with at least one counterpart:", sorted(cp.mouse.unique()), "| mice checked without one:",
      sorted(set(ok.mouse) - set(cp.mouse)))
print("r of counterparts: %.3f to %.3f; largest r of the others: %.3f" % (
      cp.r_best.min(), cp.r_best.max(), ok[ok.r_best <= 0.5].r_best.max() if (ok.r_best <= 0.5).any() else float("nan")))
print("matched dandiset mice whose OSF folders hold no full-size micrograph:",
      sorted(set(dmice) - set(ok.mouse)))

**Result: reproduced.** All 20 dandiset mice are matched by number to exactly one group folder of
the NYU component of the White Matter Microscopy Database, and no group folder holds more than 3
dandiset mice (the control group at 6 weeks holds 2: mouse 17 is not in the dandiset). Of 39 OSF
micrographs compared, 16 have a counterpart in the dandiset (r from 0.593 to 0.903), every one an
image of the same mouse, from 8 of the 11 mice checked; the others correlate at most 0.487 with any
dandiset image, as expected for the OSF images the dandiset does not hold. Nine dandiset mice
(12 and 29 to 36) have no full-size micrograph on OSF, so they are linked by number alone. The
link gives 8 control mice (7, 9, 25, 26, 27, 34, 35, 36) and 12 cuprizone mice.

## Per-image scores, and the test-set readings

The paper's test set is 8 images from each of 3 mice not used for training (Table 1). The
dandiset holds the 20 mice of the paper's training and test sets together, without saying which
3 were tested. The claims name two readings: (a) every image of the dandiset, and (b) the
images of the control mice only. The results for the cuprizone mice apart, per mouse, and, as a
variant, the range over every possible set of 3 mice (all 1140), are shown too.

In [ ]:
res["control"] = res.mouse.map(dict(zip(link.mouse, link.control)))
res["group"] = res.mouse.map(dict(zip(link.mouse, link.group)))
M = ["axon_dice", "myelin_dice", "accuracy", "sensitivity", "precision"]
EXPECTED = {"accuracy": (0.9451, 0.0945), "axon_dice": (0.9493, 0.0949), "myelin_dice": (0.8552, 0.0855),
            "sensitivity": (0.9597, 0.0960), "precision": (0.9647, 0.0965)}
readings = {"(a) all images": res, "(b) control mice": res[res.control], "cuprizone mice (not a reading)": res[~res.control]}

def summary(tag):
    rows = {}
    for name, d in readings.items():
        rows[name] = {"images": len(d), "mice": d.mouse.nunique(), **{m: d[f"{tag}_{m}"].mean() for m in M}}
    return pd.DataFrame(rows).T

per_mouse = res.groupby(["mouse", "group"])[[f"inv_{m}" for m in M]].mean().round(4)
per_mouse.columns = M
print("Per mouse, contrast inverted (segment.py):")
print(per_mouse.to_string())
mouse_means = res.groupby("mouse")[[f"inv_{m}" for m in M]].mean()
mouse_n = res.groupby("mouse").size()
combos = list(itertools.combinations(mouse_means.index, 3))
sub = {m: [] for m in M}
for c in combos:
    w = mouse_n[list(c)].values
    for m in M:
        sub[m].append(np.average(mouse_means.loc[list(c), f"inv_{m}"].values, weights=w))
ctl_mice = list(link.mouse[link.control])
ctl_combos = [c for c in combos if set(c) <= set(ctl_mice)]
print(len(combos), "sets of 3 mice;", len(ctl_combos), "of them all controls")

## table3-tem-mice-accuracy

> The TEM model trained on mouse images segments the mouse test images with a mean three-class
> pixel-wise accuracy of 95% (0.9451 in Table 3, mean over 24 test images).

Expected 0.9451, tolerance 0.0945 (0.8506 to 1.0396).

In [ ]:
lo, hi = EXPECTED["accuracy"][0] - EXPECTED["accuracy"][1], EXPECTED["accuracy"][0] + EXPECTED["accuracy"][1]
s_inv, s_no = summary("inv"), summary("noinv")
print("Contrast inverted once (segment.py):")
print(s_inv[["images", "mice", "accuracy"]].round(4).to_string())
print("Variant, no inversion:")
print(s_no[["images", "mice", "accuracy"]].round(4).to_string())
for name in ["(a) all images", "(b) control mice"]:
    v = s_inv.loc[name, "accuracy"]
    print(f"{name}: {v:.4f}, within {lo:.4f} to {hi:.4f}: {lo <= v <= hi}")
print("per-image accuracy: %.4f to %.4f (median %.4f); per-mouse means %.4f to %.4f" % (
      res.inv_accuracy.min(), res.inv_accuracy.max(), res.inv_accuracy.median(),
      per_mouse.accuracy.min(), per_mouse.accuracy.max()))
print("over all 1140 sets of 3 mice: %.4f to %.4f; over the %d sets of 3 control mice: %.4f to %.4f" % (
      min(sub["accuracy"]), max(sub["accuracy"]), len(ctl_combos),
      min(sub["accuracy"][combos.index(c)] for c in ctl_combos), max(sub["accuracy"][combos.index(c)] for c in ctl_combos)))
print("images below the lower bound %.4f: %d of %d" % (lo, int((res.inv_accuracy < lo).sum()), len(res)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
order = per_mouse.sort_values("accuracy").index
for i, (key, d) in enumerate(res.groupby("mouse")):
    ax[0].scatter(np.full(len(d), list(order.get_level_values(0)).index(key)), d.inv_accuracy,
                  c="tab:blue" if d.control.iloc[0] else "tab:orange", s=12)
ax[0].axhspan(lo, min(hi, 1.0), color="0.9", zorder=0)
ax[0].axhline(0.9451, color="k", lw=0.8)
ax[0].set_xticks(range(len(order)), [str(m) for m in order.get_level_values(0)], fontsize=7)
ax[0].set_xlabel("mouse (blue control, orange cuprizone)"); ax[0].set_ylabel("pixel-wise accuracy")
ax[0].set_ylim(0.85, 1.0)
ax[1].hist(sub["accuracy"], bins=30, color="0.5")
ax[1].axvline(0.9451, color="k", lw=0.8)
ax[1].set_xlabel("mean accuracy, every set of 3 mice"); ax[1].set_ylabel("sets")
plt.tight_layout(); plt.show()
print("left panel: %d images, accuracy %.4f to %.4f; points below 0.85 (outside the axis): %d" % (
      len(res), res.inv_accuracy.min(), res.inv_accuracy.max(), int((res.inv_accuracy < 0.85).sum())))
print("right panel: %d sets, %.4f to %.4f" % (len(sub["accuracy"]), min(sub["accuracy"]), max(sub["accuracy"])))

**Result: reproduced.** The mean pixel-wise accuracy
is 0.9416 over all 158 images (reading a) and 0.9349 over the 62 images of the 8 control mice
(reading b), both inside 0.8506 to 1.0396; the cuprizone mice give 0.9459. Every set of 3 mice,
which includes the paper's unnamed test set, gives 0.9226 to 0.9683, and no single image is below
the lower bound (lowest 0.8954), so the outcome does not depend on which mice the paper
tested. Both readings include images of the mice the model was trained
on, so the match is weaker evidence than a test on the paper's held-out mice. Without the contrast
inversion the accuracy falls to 0.5027 (a) and 0.3934 (b): the inversion that `segment.py`
applies for this model is needed.

## table3-tem-mice-dice-detection

> On the mouse test images the TEM model reaches a mean axon Dice of 0.9493, a mean myelin Dice
> of 0.8552, a mean axon detection sensitivity of 0.9597 and a mean precision of 0.9647.

Tolerance 10% of each value.

In [ ]:
q = ["axon_dice", "myelin_dice", "sensitivity", "precision"]
for name in ["(a) all images", "(b) control mice", "cuprizone mice (not a reading)"]:
    within = []
    for m in q:
        v = s_inv.loc[name, m]; e, t = EXPECTED[m]
        within.append(abs(v - e) <= t)
        print(f"{name:32s} {m:12s} {v:.4f}  expected {e} +/- {t}  within: {abs(v - e) <= t}")
    print(f"{name}: {sum(within)} of 4 within tolerance")
print("Variant, no inversion:")
print(s_no[q].round(4).to_string())
print("per-image ranges:", {m: (round(res[f'inv_{m}'].min(), 4), round(res[f'inv_{m}'].max(), 4)) for m in q})
print("per-mouse ranges:", {m: (round(per_mouse[m].min(), 4), round(per_mouse[m].max(), 4)) for m in q})
print("over all 1140 sets of 3 mice:", {m: (round(min(sub[m]), 4), round(max(sub[m]), 4)) for m in q})
for name, d in readings.items():
    print(f"{name}: ground-truth axons {int(d.inv_P.sum())}, detected (TP) {int(d.inv_TP.sum())}, false positives {int(d.inv_FP.sum())}")

**Result: reproduced.** Under reading (a) the means are axon Dice
0.9500, myelin Dice 0.8489, sensitivity 0.9863 and precision 0.9707; under reading (b) 0.9543,
0.8571, 0.9878 and 0.9819. All four are within tolerance under both readings, and over every set
of 3 mice each measure stays within its tolerance (axon Dice 0.9300 to 0.9651, myelin Dice
0.8175 to 0.8739, sensitivity 0.9722 to 0.9969, precision 0.9304 to 0.9949). Sensitivity and
precision are above the paper's values under both readings. Without the inversion all four
collapse (axon Dice 0.0029).

## myelin-dice-below-axon-dice

> In both models and every test sample the myelin Dice is lower than the axon Dice, which the
> authors attribute to myelin having two boundaries (with the axon and with the background).

Tested on the mouse images only: (i) the means of a test set, (ii) every single image.

In [ ]:
for name in ["(a) all images", "(b) control mice"]:
    d = readings[name]
    below = d.inv_myelin_dice < d.inv_axon_dice
    print(f"{name}: (i) mean myelin Dice {d.inv_myelin_dice.mean():.4f} < mean axon Dice {d.inv_axon_dice.mean():.4f}: "
          f"{d.inv_myelin_dice.mean() < d.inv_axon_dice.mean()}; (ii) single images with myelin below axon: "
          f"{int(below.sum())} of {len(d)}")
diff = res.inv_axon_dice - res.inv_myelin_dice
print("axon minus myelin Dice per image: %.4f to %.4f, median %.4f" % (diff.min(), diff.max(), diff.median()))
print("per mouse, myelin below axon in", int((per_mouse.myelin_dice < per_mouse.axon_dice).sum()), "of", len(per_mouse), "mice")

**Result: reproduced on the mouse images.** Myelin Dice is below axon Dice in the means of both
readings (0.8489 against 0.9500, and 0.8571 against 0.9543) and in every one of the 158 single
images (the difference ranges from 0.0655 to 0.1556). The SEM rows and the macaque row of
Table 3 are not part of this test.

## sensitivity-above-93

> Axon detection sensitivity is above 93% in every test set.

Tested on the mouse images only, with the mean sensitivity under each test-set reading.

In [ ]:
for name in ["(a) all images", "(b) control mice"]:
    v = readings[name].inv_sensitivity.mean()
    print(f"{name}: mean sensitivity {v:.4f}, above 0.93: {v > 0.93}")
print("images with sensitivity at or below 0.93: %d of %d (lowest %.3f)" % (
      int((res.inv_sensitivity <= 0.93).sum()), len(res), res.inv_sensitivity.min()))
print("lowest per-mouse mean %.4f; lowest over all 1140 sets of 3 mice %.4f" % (per_mouse.sensitivity.min(), min(sub["sensitivity"])))

**Result: reproduced on the mouse images.** The mean sensitivity is 0.9863 under reading (a) and
0.9878 under reading (b), above 0.93; the lowest per-mouse mean is 0.9682 and the lowest over
every set of 3 mice 0.9722. Two of the 158 single images are at or below 0.93 (lowest 0.898).
The SEM rows and the macaque row are not part of this test.

## The macaque image (data elsewhere)

The macaque TEM image of Table 3 is not in the dandiset. The White Matter Microscopy Database,
which the paper's Data availability statement names, holds the macaque corpus callosum TEM
images of ref. 7 (folder MontrealNeurologicalInstitute/histology/Macaque of OSF node yp4qg):
eight samples of three images each, 2300 x 3040 pixels (20.7 x 27.4 um at the 0.009 um of
Table 1, which gives 27 x 21 um2), and one myelin and one axon mask per sample. The paper
names neither the sample nor the image it tested. Each mask is matched here to the image of its
sample that it fits (myelin is the darkest structure, so the grey-level contrast between the
rest of the image and the myelin mask is large for the right image and near 0 for the others),
and every matched image is one reading of "the macaque test image".

In [ ]:
mfiles = ads.osf_macaque_files()
print(mfiles.to_string())
mac_rows = []
for s in sorted(mfiles["sample"].unique()):
    seg = mfiles[(mfiles.kind == "segmentation") & (mfiles["sample"] == s)].set_index("name").id
    my = ads.read_osf_nifti(seg["myelin.nii"])
    ax = ads.read_osf_nifti(seg["avf.nii"])
    m = my > 0
    m_shift = np.roll(m, (50, 50), axis=(0, 1))
    cands = []
    for _, r in mfiles[(mfiles.kind == "data") & (mfiles["sample"] == s)].iterrows():
        im = ads.read_osf_nifti(r.id)
        cands.append((im[~m].mean() - im[m].mean(), im[~m_shift].mean() - im[m_shift].mean(), r["name"], im))
    for c, cs, name, im in cands:
        print(f"{s} {name}: shape {im.shape}, grey {im.min():.0f} to {im.max():.0f}, integer valued {bool(np.all(im == np.round(im)))}; "
              f"contrast {c:.1f}, with the mask shifted by 50 px {cs:.1f}")
    print(f"{s} masks: myelin values {np.unique(my).tolist()}, axon values {np.unique(ax).round(5).tolist()}, "
          f"pixels in both {int((m & (ax > 0)).sum())}")
    c, cs, name, im = max(cands, key=lambda t: t[0])
    row = {"sample": s, "image": name, "contrast": c, "contrast_shifted": cs, "matched": c > 20}
    if row["matched"]:
        gt = ads.macaque_ground_truth(my, ax)
        for invert, tag in ((True, "inv"), (False, "noinv")):
            pred, rs_shape, n_patches = ads.segment(np.round(im).astype(np.uint8), ads.MACAQUE_PIXEL_UM, weights, config, invert=invert)
            row.update({f"{tag}_{k}": v for k, v in ads.scores(pred, gt).items()})
        row["n_patches"] = n_patches
    mac_rows.append(row)
mac = pd.DataFrame(mac_rows)
print(mac[["sample", "image", "contrast", "contrast_shifted", "matched"]].round(1).to_string())
print("samples whose masks fit one of their images:", int(mac.matched.sum()), "of", len(mac))

## table3-tem-macaque-accuracy

> On the macaque corpus callosum TEM test image the mouse-trained TEM model reaches a pixel-wise
> accuracy of 84% (0.8438).

Expected 0.8438, tolerance 0.0844 (0.7594 to 0.9282). One reading per matched image.

In [ ]:
mm = mac[mac.matched].set_index("sample")
lo_m, hi_m = 0.8438 - 0.0844, 0.8438 + 0.0844
cols = ["image", "inv_accuracy", "inv_axon_dice", "inv_myelin_dice", "inv_sensitivity", "inv_precision", "noinv_accuracy"]
print(mm[cols].round(4).to_string())
within = (mm.inv_accuracy >= lo_m) & (mm.inv_accuracy <= hi_m)
print(f"readings within {lo_m:.4f} to {hi_m:.4f}: {int(within.sum())} of {len(mm)} ({', '.join(mm.index[within])})")
print("accuracy over the readings: %.4f to %.4f, mean %.4f" % (mm.inv_accuracy.min(), mm.inv_accuracy.max(), mm.inv_accuracy.mean()))
print("Table 3, macaque row, for the other measures: axon Dice 0.9069, myelin Dice 0.7519, sensitivity 0.9429, precision 0.8129")
for m in ["axon_dice", "myelin_dice", "sensitivity", "precision"]:
    print(f"  {m}: {mm['inv_' + m].min():.4f} to {mm['inv_' + m].max():.4f}")
print("ground-truth axon objects per image:", mm.inv_P.astype(int).to_dict(), "| patches per image:", mm.n_patches.astype(int).to_dict())

**Result: blocked (methods underspecified).** The masks of 7 of the 8 macaque samples fit one of
their images; those of sample 4 fit none of its three images. On the 7 matched images the
pixel-wise accuracy ranges from 0.5729 to 0.7634: one reading (sample 3) is inside 0.7594 to
0.9282 and six are below it, so the readings disagree, and the paper does not say which image it
tested. The other measures are below Table 3's macaque row on every image (axon Dice 0.6268 to
0.8070 against 0.9069, myelin Dice 0.5032 to 0.6481 against 0.7519, sensitivity 0.289 to 0.606
against 0.9429). On every one of the 7 images the myelin Dice is below the axon
Dice. Without the inversion the accuracy is 0.1810 to 0.3540.

## macaque-myelin-underestimated

> The TEM model slightly underestimates the myelin sheaths of the macaque sample compared with
> its ground truth.

The criterion's reading: the predicted myelin area is below the ground-truth myelin area.

In [ ]:
ratio = mm.inv_pred_myelin_frac / mm.inv_gt_myelin_frac
print(pd.DataFrame({"pred_myelin_frac": mm.inv_pred_myelin_frac, "gt_myelin_frac": mm.inv_gt_myelin_frac,
                    "ratio": ratio, "pred_axon_frac": mm.inv_pred_axon_frac, "gt_axon_frac": mm.inv_gt_axon_frac}).round(3).to_string())
print("readings with predicted myelin area below the ground truth:", int((ratio < 1).sum()), "of", len(mm),
      "| ratio %.3f to %.3f" % (ratio.min(), ratio.max()))
mouse_ratio = res.inv_pred_myelin_frac / res.inv_gt_myelin_frac
print("for comparison, mouse images: ratio %.3f to %.3f, median %.3f" % (mouse_ratio.min(), mouse_ratio.max(), mouse_ratio.median()))

**Result: reproduced.** On all 7 matched macaque images the predicted myelin area is below the
ground truth's. The underestimate is large: the predicted myelin fraction is 0.352 to 0.512 of
the ground truth's, against 0.788 to 1.266 (median 0.974) on the mouse images, so the paper's
"slightly" does not hold on these images.

## Claims not tested

The dandiset holds only the mouse TEM images and their masks. The SEM images of rat and human
spinal cord (Table 3, rows Rat 1, Rat 2, Human) and the full rat cervical spinal cord slice of
Figures 6 and 7 are not in it. They were looked for in the White Matter Microscopy Database (OSF
node yp4qg), which the paper names: its rat spinal cord images with ground truth
(PolytechniqueMontreal/RatSpinalCordNov2015) are CARS images at 0.225 to 0.33 um, its human spinal
cord images (PolytechniqueMontreal/Human) are at 0.23 um, and neither matches the SEM test images of
Table 1 (0.13, 0.07 and 0.13 um). It also holds an SEM panorama of an adult rat cervical spinal cord
(PolytechniqueMontreal/RatSpinalCordNov2016, 0.095 um, 20006 x 18255 pixels), but the paper gives
no pixel size, date or deposit for the slice of Figures 6 and 7, so it cannot be shown to be that
slice. These claims are not tested here.

In [ ]:
print("species in participants.tsv:", sorted(set(pd.read_csv(pd.io.common.BytesIO(
      ads.read_bytes(assets.set_index('path').asset_id['participants.tsv'])), sep='\t').species)))
print("modalities in the file names:", sorted(set(assets.path.str.extract(r'_(TEM|SEM|BF|[A-Z]+)\.png$')[0].dropna())))
print("body part in the sidecars:", sorted({d['BodyPart'] + '/' + d['BodyPartDetails'] for d in sidecars.values()}))

## Summary

| Claim | Outcome |
|---|---|
| osf-group-labels-link (registry check) | reproduced |
| table3-tem-mice-accuracy | reproduced |
| table3-tem-mice-dice-detection | reproduced |
| myelin-dice-below-axon-dice | reproduced on the mouse images |
| sensitivity-above-93 | reproduced on the mouse images |
| table3-tem-macaque-accuracy | blocked, methods underspecified (which image) |
| macaque-myelin-underestimated | reproduced (direction; the size is not slight) |
| table3-sem-rat-accuracy, table3-sem-human-accuracy | blocked, data missing |
| table3-other-samples-dice-detection | blocked, data missing (SEM); macaque values shown above |
| full-slice-processing-time, fig7-corticospinal-tract, fig7-larger-axons-periphery, fig7-tract-diameters, fig7-gratio-range | blocked, data missing |

How the analysis differs from the paper's. The paper's network runs in TensorFlow 1, which cannot
be installed with Python 3.13; it is rebuilt here from the released checkpoint, and its output
matches the model's own saved graph to within 1e-5 on a test image. Resampling uses the settings
of scikit-image 0.13. The paper's test set (3 held-out mice) is not named, so the mouse claims
are tested on all mice and on the control mice, both of which include training mice, with every
set of 3 mice shown. The dandiset's masks are those of 2025 (fibres separated by a 2-pixel
line), not the masks the paper scored against. The macaque images and masks come from the White
Matter Microscopy Database, and which of them the paper tested is not known.

What was found about the dataset. The dandiset holds 158 TEM images of 20 mice (8 per mouse,
6 for mouse 25) with three masks each, and the masks fit their images (myelin contrast median
84.4 grey levels in place against 11.8 when shifted). No image is deposited twice. The pixel size
is 0.00236 um, not the 0.002 um of the paper (the README says so; at 0.002 um the images would be
too small for the model's 512-pixel patches). The files do not say which mice are controls; the
White Matter Microscopy Database does, by mouse number. The archive's summary counts 11 samples
where there are 158 images (sample ids repeat across mice). 21 axonmyelin masks are grey plus
alpha pngs, the rest grey. The test split of the paper is not recorded.